In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
DROPPED_COL = ["passengerid", "cabin", "ticket", "embarked", "name"]

In [ ]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')


def encode_OHE(X, fit=True):
    categorical_columns = X.select_dtypes("object").columns

    one_hot_encoded = encoder.fit_transform(
        X[categorical_columns]) if fit else encoder.transform(X[categorical_columns])

    one_hot_X = pd.DataFrame(
        one_hot_encoded, columns=encoder.get_feature_names_out(categorical_columns))

    X_encoded = pd.concat([X.reset_index(
        drop=True), one_hot_X.reset_index(drop=True)], axis=1)

    X_encoded = X_encoded.drop(categorical_columns, axis=1)
    return X_encoded

## EDA

In [ ]:
df_train = pd.read_csv(r"datasets\train.csv")
df_test = pd.read_csv(r"datasets\test.csv")

In [ ]:
print(f"Shape of train: {df_train.shape}")
print(f"Shape of test: {df_test.shape}")

In [ ]:
df_train.head()

In [ ]:
df_train = df_train.rename(columns={col: col.lower()
                           for col in df_train.columns})
df_test = df_test.rename(columns={col: col.lower() for col in df_test.columns})

In [ ]:
X = df_train.drop(["survived", *DROPPED_COL], axis=1)
y = df_train["survived"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.214, random_state=42, stratify=y)

In [ ]:
print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")

In [ ]:
print(
    f"Number of Survived vs Not Survived:\n{y_train.value_counts(normalize=True)}")

In [ ]:
categorical_cols = X_train.select_dtypes(include="object").columns
numerical_cols = X_train.select_dtypes(include="number").columns
print(f"Categorical Columns ({len(categorical_cols)}): {categorical_cols}")
print(f"Numerical Columns ({len(numerical_cols)}): {numerical_cols}")

In [ ]:
X_train.info()

In [ ]:
X_train.describe(include='all').T

## Vis

In [ ]:
corr = pd.concat([X_train, y_train], axis=1).corr(numeric_only=True)
mask = np.zeros_like(corr)

mask[np.triu_indices_from(mask)] = True
sns.heatmap(corr, annot=True, cmap="coolwarm",
            mask=mask | (np.abs(corr) <= 0.1))

In [ ]:

_ = X_train.hist(bins=50)
plt.tight_layout()

## Remove Null

In [ ]:
X_train.isna().sum()[lambda x:x > 0] / len(X_train)

In [ ]:
# X_train["embarked"] = X_train["embarked"].fillna("S")

age_med = X_train["age"].median()
X_train["age"] = X_train["age"].fillna(age_med)

## Remove outliers

In [ ]:
sns.boxplot(X_train["fare"])

## Extract Title & Create New Feature

In [ ]:
# X_train["tname"] = X_train["name"].str.extract(r"(\w{2,})\.")
# X_train = X_train.drop(["name"], axis=1)

In [ ]:
X_train["tfamily"] = X_train["sibsp"] + X_train["parch"] + 1
X_train["isalone"] = X_train["tfamily"] == 1

X_train["fare"] = np.log1p(X_train["fare"])

## Encode Categories

In [ ]:
X_train["sex"] = (X_train["sex"] == "male")

In [ ]:
X_train["sex"].value_counts()

In [ ]:
# X_train = encode_OHE(X_train)
# X_train.head()

## Train Model

In [ ]:
rfc = RandomForestClassifier()
rfc.fit(X_train, y_train)

## Measure Performance

In [ ]:
y_hat = rfc.predict(X_train)
print(f"F1 Score for Train: {f1_score(y_train, y_hat)}")

cross_val_score(rfc, X_train, y_train, scoring="f1")

In [ ]:
cm = confusion_matrix(y_train, y_hat, labels=rfc.classes_)

disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=rfc.classes_)
disp.plot()

## Prepare df_test

In [ ]:
# drop not needed cols
df_test = df_test.drop([*DROPPED_COL], axis=1)

# fill null for age
df_test["age"] = df_test["age"].fillna(age_med)

# Extract title from names
# df_test["tname"] = df_test["name"].str.extract(r"(\w{2,})\.")
# df_test = df_test.drop(["name"], axis=1)

# Create new feature
df_test["tfamily"] = df_test["sibsp"] + df_test["parch"] + 1
df_test["isalone"] = df_test["tfamily"] == 1

# fix skewness
df_test["fare"] = np.log1p(df_test["fare"])

# encode sex to true and false
df_test["sex"] = (df_test["sex"] == "male")

# encdoe categorical cols
# df_test = encode_OHE(df_test, fit=False)

In [ ]:
# create passenger_id start from 892 for df_test
passenger_id = df_test.index+892
# predict Survived from df_test
y_test_predicted = rfc.predict(df_test)

# create df_test submission
pd.DataFrame({"PassengerId": passenger_id,
             "Survived": y_test_predicted}).to_csv("base_submit.csv", index=False)